In [23]:
import io
import requests
import pandas as pd

r0 = requests.post(
    "https://api.lsst.fink-portal.org/api/v1/tags",
    json={
        "tag": "sn_near_galaxy_candidate",
        "n": "10",
        "columns": "r:diaObjectId",  
    },
)

mylist = [str(val["r:diaObjectId"]) for val in r0.json()]

# get full lightcurves for all these alerts
r1 = requests.post(
    "https://api.lsst.fink-portal.org/api/v1/sources",
    json={
       "diaObjectId": ",".join(mylist),
       "columns": "r:diaObjectId,r:apFlux,r:psfFlux,r:psfFluxErr",  
       "output-format": "json",
   },
)

# Format output in a DataFrame
pdf = pd.read_json(io.BytesIO(r1.content))
print (pdf)

        r:apFlux       r:diaObjectId  r:psfFlux  r:psfFluxErr
0     5738.52100  313871013236965412  5416.7610     286.09982
1     1435.08000  313871015699021854  1520.3877     116.14518
2     3504.67140  313888627086590051  3989.4272     279.25314
3     5895.20400  313871013236965412  5504.8560     263.06195
4     3777.59450  313888627086590051  3608.4822     244.40830
...          ...                 ...        ...           ...
3699   747.10425  314011776610992165  1506.2758     241.53970
3700  6520.66200  313871013236965412  5468.8213     286.65536
3701   328.01788  313871013107990657  1189.4263     155.60905
3702  3438.20340  313871013236965412  4221.2830     387.90146
3703  1609.02970  314011776610992165  1823.4150     279.95790

[3704 rows x 4 columns]


In [12]:
import io
import requests
import pandas as pd

# Get latests 5 alerts reported to TNS
r = requests.post(
    "https://api.lsst.fink-portal.org/api/v1/tags",
    json={
        "tag": "in_tns",
        "columns": "r:diaObjectId",
        "n": "5",  
    },
)

# Format output in a DataFrame
pdf = pd.read_json(io.BytesIO(r.content))
print (pdf)

        r:diaObjectId
0  170226393632735260
1  313871013236965412
2  313893021775560722
3  313998569623257167
4  314003014107006318


In [31]:
import requests as re
import pandas as pd
import os
from datetime import datetime

# API Configuration
API_URL_TAGS = "https://api.lsst.fink-portal.org/api/v1/tags"
API_URL_SOURCES = 'https://api.lsst.fink-portal.org/api/v1/sources'
SAVE_FILE = "sn_ia_candidates.csv"

# Assessment Thresholds
MIN_EARLY_SNIA_SCORE = 0  # Confidence in Early SN Ia
MIN_SNN_SCORE = 0         # SuperNNova SN vs Others score
CATS_SN_CLASS = 11           # CATS classifier code for SN-like

def dia_object_collection(n=1000):
    r0 = re.post(API_URL_TAGS,
        json={
            "tag": "in_tns",
            "n": n,
            "columns": "r:diaObjectId",  
        },
    )
    mylist = [str(val["r:diaObjectId"]) for val in r0.json()]
    return mylist

def fetch_latest_alerts():
    """Fetches the latest alerts from the Fink LSST stream."""
    params = {'diaObjectId': ','.join(mylist),
            'columns':"r:diaObjectId,r:midpointMjdTai,r:psfFlux,r:psfFluxErr,r:clf_cats_class,r:clf_earlySNIa_score,r:clf_snnSnVsOthers_score",
            'output-format': 'json' }
    
    response = re.post(API_URL_SOURCES, json=params)
    if response.status_code == 200:
        return pd.DataFrame(response.json())
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

def assess_candidates(df):
    """Filters alerts based on SN Ia likelihood scores."""
    # Ensure relevant columns exist (LSST schema uses clf_ prefix)
    required_cols = ['r:diaObjectId','r:midpointMjdTai', 'r:psfFlux''r:psfFluxErr','r:clf_earlySNIa_score','r:clf_cats_class','r:clf_snnSnVsOthers_score']
    for col in required_cols:
        if col not in df.columns:
            # Add dummy 0s if the classifier hasn't run on this batch
            df[col] = df.get(col, 0)

    # Filtering logic:
    # 1. High Early SN Ia score OR
    # 2. CATS class is 11 (SN-like) and SuperNNova score is high
    mask = (
        (df['r:clf_earlySNIa_score'] >= MIN_EARLY_SNIA_SCORE) |
        ((df['r:clf_cats_class'] == CATS_SN_CLASS) & (df['r:clf_snnSnVsOthers_score'] >= MIN_SNN_SCORE))
    )
    
    candidates = df[mask].copy()
    
    # Clean up: Add a timestamp of when we assessed it
    candidates['assessment_date'] = datetime.now().strftime("%Y-%m-%d %H:%M")
    
    return candidates[['r:diaObjectId','r:midpointMjdTai', 'r:psfFlux''r:psfFluxErr','r:clf_earlySNIa_score', 'r:clf_cats_class', 'r:clf_snnSnVsOthers_score', 'assessment_date']]

def save_candidates(new_candidates):
    """Saves object IDs to a CSV without duplicates."""
    if os.path.exists(SAVE_FILE):
        existing = pd.read_csv(SAVE_FILE)
        updated = pd.concat([existing, new_candidates]).drop_duplicates(subset=['objectId'])
    else:
        updated = new_candidates
        
    updated.to_csv(SAVE_FILE, index=False)
    print(f"Saved {len(new_candidates)} potential candidates. Total unique: {len(updated)}")

if __name__ == "__main__":
    print("Polling Fink LSST for new SN Ia candidates...")
    alerts_df = fetch_latest_alerts()
    
    if alerts_df is not None and not alerts_df.empty:
        hits = assess_candidates(alerts_df)
        if not hits.empty:
            save_candidates(hits)
        else:
            print("No candidates met the threshold in this batch.")

Polling Fink LSST for new SN Ia candidates...
Saved 3704 potential candidates. Total unique: 2


In [36]:
import requests
import pandas as pd

# LSST Fink API Endpoints
TAG_ENDPOINT = "https://api.lsst.fink-portal.org/api/v1/tags"
OBJECT_ENDPOINT = "https://api.lsst.fink-portal.org/api/v1/objects"

def get_latest_candidates(tag='sn_near_galaxy_candidate', n=100):
    """Step 1: Pull the latest diaObjectIds associated with a specific science tag."""
    payload = {
        'tag': tag,
        'n': n,
        'output-format': 'json'
    }
    r = requests.post(TAG_ENDPOINT, json=payload)
    if r.status_code == 200:
        return [item['r:diaObjectId'] for item in r.json()]
    return []

def assess_for_excess(dia_id):
    """Step 2 & 3: Pull object data and check for early blue flux properties."""
    payload = {
        'objectId': dia_id,
        'output-format': 'json'
    }
    # Getting the object summary (aggregated features)
    r = requests.post(OBJECT_ENDPOINT, json=payload)
    if r.status_code != 200:
        return None
    
    data = r.json()[0] # The most recent record for this ID
    
    # LSST Field Names (using nJy for flux)
    # nDiaSources = number of detections
    # g_psfFluxMean = mean flux in g band
    # r_psfFluxMean = mean flux in r band
    
    n_det = data.get('nDiaSources', 0)
    g_flux = data.get('g_psfFluxMax', 0)
    r_flux = data.get('r_psfFluxMax', 0)

    # CRITERIA FOR HONOURS PROJECT:
    # 1. Catch it early (less than 5 detections)
    # 2. Blue excess (Flux g > Flux r in the early phase)
    # Note: In LSST nJy, higher flux = brighter.
    is_early = 1 <= n_det <= 5
    is_blue = g_flux > (1.2 * r_flux) # Looking for g-heavy early peaks
    
    if is_early and is_blue:
        return {
            'diaObjectId': dia_id,
            'n_detections': n_det,
            'g_flux_max': g_flux,
            'r_flux_max': r_flux,
            'ra': data.get('ra'),
            'dec': data.get('dec')
        }
    return None

if __name__ == "__main__":
    print(f"Scanning 'sn_near_galaxy_candidate' stream...")
    candidate_ids = get_latest_candidates()
    
    saved_candidates = []
    for dia_id in candidate_ids:
        result = assess_for_excess(dia_id)
        if result:
            saved_candidates.append(result)
    
    if saved_candidates:
        df_hits = pd.DataFrame(saved_candidates)
        df_hits.to_csv("early_excess_log.csv", mode='a', index=False)
        print(f"Success: Found {len(df_hits)} candidates with potential early blue excess.")
    else:
        print("No new objects met the early excess criteria.")

Scanning 'sn_near_galaxy_candidate' stream...
No new objects met the early excess criteria.
